# 🔗 Workflow Integration - Complete SAR Processing System

Welcome to Phase 4 of the Financial Services Agentic AI Project!

In this notebook, you'll integrate both AI agents into a complete **end-to-end SAR processing workflow** that demonstrates real-world financial compliance automation.

## 🎯 Learning Objectives
- Build a complete two-stage AI workflow with human oversight
- Implement human-in-the-loop decision gates for compliance
- Generate complete SAR documents from AI analysis
- Create comprehensive audit trails for regulatory examination
- Demonstrate cost optimization through intelligent agent coordination

## 📋 Business Context
This workflow simulates how banks actually process suspicious activity reports:
1. **Risk Screening**: AI agents analyze transaction patterns for suspicious activity
2. **Human Review**: Compliance officers review AI findings before proceeding
3. **Narrative Generation**: Only approved cases get full compliance documentation
4. **SAR Filing**: Complete regulatory forms are generated for submission
5. **Audit Documentation**: Every decision is logged for regulatory examination

## 🏗️ System Architecture

```
📊 CSV Data → 🔍 Risk Analyst → 👤 Human Decision → ✅ Compliance Officer → 📄 SAR Document
              (Chain-of-Thought)    (Gate)         (ReACT Framework)     (FinCEN Ready)
```

## 🚀 Prerequisites Check

Before starting, ensure you have completed:
- ✅ Phase 1: Foundation components (`foundation_sar.py`)
- ✅ Phase 2: Risk Analyst Agent (`risk_analyst_agent.py`)
- ✅ Phase 3: Compliance Officer Agent (`compliance_officer_agent.py`)
- ✅ Both agents pass their comprehensive test scenarios

If any component is missing, return to previous notebooks to complete implementation.

In [ ]:
# Setup and Environment Configuration
import os
import sys
import json
import pandas as pd
import uuid
import hashlib
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv

# Add project root + src so `src` package is importable
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
src_path = os.path.join(project_root, "src")
for path in [project_root, src_path]:
    if path not in sys.path:
        sys.path.insert(0, path)

# Load environment variables
load_dotenv(os.path.join(project_root, ".env"))

print("📚 Libraries imported successfully!")
print("🔐 Environment variables loaded")
print(f"📂 Project root on path: {project_root}")
print(f"📂 Src path on path: {src_path}")

In [ ]:
# OpenAI Setup for Vocareum
import openai

# Initialize OpenAI client for Vocareum
openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
    print("Get your Vocareum OpenAI API key from 'Cloud Resources' in your workspace")
else:
    # Vocareum requires routing through their servers
    client = openai.OpenAI(
        base_url="https://openai.vocareum.com/v1",
        api_key=openai_api_key
    )
    print("✅ OpenAI client initialized with Vocareum routing")
    print(f"🔑 API key: {openai_api_key[:8]}...{openai_api_key[-4:]}")
    print("📍 Base URL: https://openai.vocareum.com/v1")

In [ ]:
# Import your implemented components
from src.foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data,
)
from src.risk_analyst_agent import RiskAnalystAgent
from src.compliance_officer_agent import ComplianceOfficerAgent

# Create outputs folders
os.makedirs("../outputs/audit_logs", exist_ok=True)
os.makedirs("../outputs/filed_sars", exist_ok=True)

# Create agent instances
explainability_logger = ExplainabilityLogger("../outputs/audit_logs/workflow_integration.jsonl")
risk_agent = RiskAnalystAgent(client, explainability_logger)
compliance_agent = ComplianceOfficerAgent(client, explainability_logger)

print("✅ Components imported and agents initialized")

## 📊 Step 1: Data Loading and Preprocessing

Load the financial data and prepare it for analysis.

In [ ]:
# Load and Preprocess Financial Data

def load_and_preprocess_data():
    """
    Load CSV data and prepare for analysis.

    Steps:
    1. Load customers.csv, accounts.csv, transactions.csv
    2. Normalize missing values to None
    3. Convert DataFrames to dict records
    """
    print("📊 Loading Financial Data")

    customers_df = pd.read_csv("../data/customers.csv", dtype={"ssn_last_4": str})
    accounts_df = pd.read_csv("../data/accounts.csv")
    transactions_df = pd.read_csv("../data/transactions.csv")

    # Normalize NaN values to None for optional fields
    customers_df = customers_df.where(pd.notna(customers_df), None)
    accounts_df = accounts_df.where(pd.notna(accounts_df), None)
    transactions_df = transactions_df.where(pd.notna(transactions_df), None)

    customers_data = customers_df.to_dict("records")
    accounts_data = accounts_df.to_dict("records")
    transactions_data = transactions_df.to_dict("records")

    print(
        f"📈 Loaded: {len(customers_data)} customers, "
        f"{len(accounts_data)} accounts, {len(transactions_data)} transactions"
    )
    return customers_data, accounts_data, transactions_data

# Load data
customers_data, accounts_data, transactions_data = load_and_preprocess_data()

## 🎯 Step 2: Customer Risk Screening

Implement intelligent customer screening to identify high-risk cases for detailed analysis.

In [ ]:
# Implement Customer Risk Screening

def screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=5):
    """
    Risk-based screening to select top N customers for analysis.

    Criteria:
    1. High risk ratings (Medium, High)
    2. Large total transaction volume (>$100K)
    3. High transaction frequency (>50)
    """
    print("🔍 Customer Risk Screening")

    selected_customers = []

    for customer in customers_data:
        customer_accounts = [
            acc for acc in accounts_data if acc.get("customer_id") == customer.get("customer_id")
        ]
        account_ids = {acc.get("account_id") for acc in customer_accounts}
        customer_transactions = [
            txn for txn in transactions_data if txn.get("account_id") in account_ids
        ]

        total_amount = sum(abs(txn.get("amount", 0) or 0) for txn in customer_transactions)
        transaction_count = len(customer_transactions)
        risk_rating = customer.get("risk_rating")

        risk_flags = []
        if risk_rating in ["Medium", "High"]:
            risk_flags.append("high_risk_rating")
        if total_amount > 100000:
            risk_flags.append("large_amounts")
        if transaction_count > 50:
            risk_flags.append("high_frequency")

        if len(risk_flags) >= 2:
            selected_customers.append(
                {
                    "customer": customer,
                    "accounts": customer_accounts,
                    "transactions": customer_transactions,
                    "total_amount": total_amount,
                    "transaction_count": transaction_count,
                    "risk_flags": risk_flags,
                }
            )

    selected_customers.sort(
        key=lambda x: (len(x["risk_flags"]), x["total_amount"]), reverse=True
    )
    selected = selected_customers[:top_n]

    print(f"📊 Selected {len(selected)} customers for analysis")
    return selected

# Run customer screening
selected_customers = screen_high_risk_customers(customers_data, accounts_data, transactions_data)

## 🤖 Step 3: Two-Stage AI Analysis with Human Gates

Implement the core two-stage workflow:
1. **Stage 1**: Risk Analyst performs Chain-of-Thought analysis
2. **Human Gate**: Review and decision to proceed
3. **Stage 2**: Compliance Officer generates ReACT narratives (only if approved)

In [ ]:
# Implement Two-Stage AI Workflow

def run_two_stage_sar_workflow(selected_customers, auto_approve=False):
    """
    Complete two-stage SAR processing workflow.

    For each customer:
    1. Create CaseData object
    2. Run Risk Analyst analysis
    3. Present findings to human reviewer
    4. Get human decision (proceed/reject)
    5. If approved: Run Compliance Officer narrative
    6. Generate SAR document
    7. Log decisions for audit
    """
    print("🤖 Two-Stage SAR Processing Workflow")

    processed_cases = []
    approved_sars = []
    rejected_cases = []
    audit_decisions = []

    loader = DataLoader(explainability_logger)

    for i, customer_data in enumerate(selected_customers, 1):
        print(f"\n🔍 CUSTOMER {i}/{len(selected_customers)}: {customer_data['customer']['name']}")
        print("=" * 60)

        try:
            case_data = loader.create_case_from_data(
                customer_data["customer"],
                customer_data["accounts"],
                customer_data["transactions"],
            )

            print("🔍 STAGE 1: Risk Analysis")
            risk_analysis = risk_agent.analyze_case(case_data)

            print(f"Classification: {risk_analysis.classification}")
            print(f"Confidence: {risk_analysis.confidence_score:.2f}")
            print(f"Risk Level: {risk_analysis.risk_level}")
            print(f"Reasoning: {risk_analysis.reasoning}")

            if auto_approve:
                decision = "yes"
            else:
                decision = input("🤔 Proceed with SAR filing? (yes/no): ").strip().lower()
            should_proceed = decision in ["yes", "y"]

            if should_proceed:
                print("📝 STAGE 2: Compliance Narrative Generation")
                compliance_review = compliance_agent.generate_compliance_narrative(
                    case_data, risk_analysis
                )

                sar_document = create_sar_document(
                    case_data, risk_analysis, compliance_review
                )
                save_sar_document(sar_document)
                approved_sars.append(sar_document)
                print(f"✅ SAR FILED: {sar_document['sar_metadata']['sar_id']}")
            else:
                rejected_cases.append(
                    {"case_id": case_data.case_id, "reason": "human_rejection"}
                )
                print("❌ SAR REJECTED by human reviewer")

            processed_cases.append(case_data)
            audit_decisions.append(
                {
                    "case_id": case_data.case_id,
                    "customer_name": case_data.customer.name,
                    "decision": "PROCEED" if should_proceed else "REJECT",
                    "ai_classification": risk_analysis.classification,
                    "ai_confidence": risk_analysis.confidence_score,
                    "reviewer_decision": decision,
                }
            )
        except Exception as exc:
            print(f"❌ Error processing customer: {exc}")

    return processed_cases, approved_sars, rejected_cases, audit_decisions

# Workflow execution happens after SAR document helpers are defined.

## 📄 Step 4: SAR Document Generation

Create complete, FinCEN-ready SAR documents with all required metadata.

In [ ]:
# Implement SAR Document Generation

def create_sar_document(case_data, risk_analysis, compliance_review):
    """
    Create a complete SAR document for regulatory submission.
    """
    print("📄 Creating SAR Document")

    sar_id = f"SAR_{uuid.uuid4()}"
    filing_date = datetime.now(timezone.utc).isoformat()

    sar_document = {
        "sar_metadata": {
            "sar_id": sar_id,
            "filing_date": filing_date,
            "filing_type": "Suspicious Activity Report",
            "ai_generated": True,
            "review_status": "human_approved",
        },
        "subject_information": {
            "customer_name": case_data.customer.name,
            "customer_id": case_data.customer.customer_id,
            "address": case_data.customer.address,
            "customer_since": case_data.customer.customer_since,
            "risk_rating": case_data.customer.risk_rating,
        },
        "suspicious_activity": {
            "classification": risk_analysis.classification,
            "risk_level": risk_analysis.risk_level,
            "confidence_score": risk_analysis.confidence_score,
            "narrative": compliance_review.narrative,
            "key_indicators": risk_analysis.key_indicators,
            "ai_reasoning": risk_analysis.reasoning,
        },
        "regulatory_compliance": {
            "citations": getattr(compliance_review, "regulatory_citations", []),
            "narrative_word_count": len(compliance_review.narrative.split()),
            "compliance_status": "approved",
        },
        "audit_trail": {
            "case_id": case_data.case_id,
            "processing_date": filing_date,
            "ai_agents_used": ["RiskAnalyst", "ComplianceOfficer"],
            "human_reviewer": "compliance_officer",
        },
    }

    return sar_document


def save_sar_document(sar_document):
    """Save SAR document to outputs directory."""
    os.makedirs("../outputs/filed_sars", exist_ok=True)
    filename = f"../outputs/filed_sars/{sar_document['sar_metadata']['sar_id']}.json"
    with open(filename, "w", encoding="utf-8") as file_handle:
        json.dump(sar_document, file_handle, indent=2)

print("📄 SAR document generation functions defined")

# Run the complete workflow now that SAR helpers exist
processed_cases, approved_sars, rejected_cases, audit_decisions = run_two_stage_sar_workflow(
    selected_customers
)

## 📊 Step 5: Workflow Metrics and Analysis

Analyze the efficiency and effectiveness of your AI-powered SAR processing system.

In [ ]:
# Workflow Analysis and Metrics

def analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions):
    """
    Calculate workflow efficiency metrics.
    """
    print("📊 Workflow Efficiency Analysis")

    total_cases = len(processed_cases)
    approved_cases = len(approved_sars)
    rejected_cases_count = len(rejected_cases)

    if total_cases > 0:
        approval_rate = approved_cases / total_cases
        rejection_rate = rejected_cases_count / total_cases
    else:
        approval_rate = rejection_rate = 0

    print("📈 WORKFLOW METRICS:")
    print(f"   Total Cases Processed: {total_cases}")
    print(f"   SARs Filed: {approved_cases}")
    print(f"   Cases Rejected: {rejected_cases_count}")
    print(f"   Approval Rate: {approval_rate:.1%}")
    print(f"   Rejection Rate: {rejection_rate:.1%}")

    print("\n💰 COST OPTIMIZATION:")
    print("   Two-stage processing saves costs by only running")
    print("   compliance generation on approved cases")
    print(f"   Cost savings: {rejection_rate:.1%} of compliance calls avoided")


def validate_ai_decisions(audit_decisions):
    """Analyze AI decision patterns and reviewer outcomes."""
    print("📋 AI Decision Review")
    if not audit_decisions:
        print("   No audit decisions recorded.")
        return

    decisions = [entry["decision"] for entry in audit_decisions]
    proceed_count = decisions.count("PROCEED")
    reject_count = decisions.count("REJECT")

    print(f"   Human approvals: {proceed_count}")
    print(f"   Human rejections: {reject_count}")

# Run analysis
analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions)
validate_ai_decisions(audit_decisions)

## 🏁 Step 6: Complete System Demonstration

Test your complete system with comprehensive scenarios to validate production readiness.

In [ ]:
# Complete System Test (optional)
# Students: Demonstrate your complete SAR processing system

def demonstrate_complete_system(run_demo=False):
    """
    Run complete system demonstration.

    This should:
    1. Process multiple customers through the complete workflow
    2. Show both approved and rejected cases
    3. Generate multiple SAR documents
    4. Demonstrate audit trail creation
    5. Show efficiency metrics
    """
    print("🏁 Complete SAR Processing System Demonstration")

    if not run_demo:
        print("📋 Set run_demo=True to execute the full demonstration")
        return

    print("🚀 Running complete system test...")

    customers_data, accounts_data, transactions_data = load_and_preprocess_data()
    selected_customers = screen_high_risk_customers(
        customers_data, accounts_data, transactions_data, top_n=3
    )
    processed_cases, approved_sars, rejected_cases, audit_decisions = (
        run_two_stage_sar_workflow(selected_customers)
    )

    analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions)

    print("🎉 System demonstration complete!")
    print("📄 SAR documents saved to: ../outputs/filed_sars/")
    print("📊 Audit logs saved to: ../outputs/audit_logs/")

# Call with run_demo=True when ready
demonstrate_complete_system(run_demo=True)

## Project Walkthrough (Quick)

This run demonstrates the full SAR pipeline:

1. Load CSV data and normalize missing values.
2. Screen for high-risk customers using volume, frequency, and risk rating.
3. Run Risk Analyst (Chain-of-Thought) to classify suspicious activity.
4. Human gate decides whether to proceed with SAR filing.
5. Run Compliance Officer (ReACT) to generate the SAR narrative.
6. Save SAR documents and audit logs.
7. Summarize outcomes and efficiency metrics.

Outputs:
- Audit trail: `../outputs/audit_logs/workflow_integration.jsonl`
- SAR documents: `../outputs/filed_sars/`


## 📝 Implementation Checklist

### ✅ Workflow Integration Deliverables
- [ ] **Data Loading**: Load and preprocess CSV data with proper error handling
- [ ] **Customer Screening**: Implement risk-based screening to identify high-risk cases
- [ ] **Two-Stage Workflow**: Build complete Risk Analyst → Human Gate → Compliance Officer flow
- [ ] **Human Decision Gates**: Implement interactive approval/rejection points
- [ ] **SAR Document Generation**: Create complete FinCEN-ready documents with metadata
- [ ] **Audit Trail Creation**: Log all decisions and reasoning for regulatory examination
- [ ] **Efficiency Metrics**: Calculate cost optimization and processing efficiency
- [ ] **System Demonstration**: Test complete workflow with multiple scenarios

### ✅ Testing and Validation Requirements
- [ ] **Component Validation**: Verify all foundation components and agents are available
- [ ] **Integration Testing**: Run comprehensive test suites for all components with proper sys.path setup
- [ ] **End-to-End Testing**: Test complete workflow with automated scenarios
- [ ] **Error Handling Testing**: Validate graceful handling of edge cases and failures
- [ ] **Output Validation**: Ensure SAR documents meet regulatory standards
- [ ] **Performance Testing**: Measure workflow efficiency and processing times

### ✅ Technical Requirements
- [ ] **Error Handling**: Robust exception handling for all workflow steps
- [ ] **Data Validation**: Proper validation of all inputs and outputs
- [ ] **File Management**: Organize outputs in appropriate directories
- [ ] **Logging**: Comprehensive audit logging for compliance
- [ ] **Performance**: Efficient processing of multiple cases
- [ ] **User Experience**: Clear prompts and feedback for human reviewers
- [ ] **Test Infrastructure**: Proper test imports and sys.path configuration

### ✅ Business Requirements  
- [ ] **Regulatory Compliance**: Ensure all SAR documents meet FinCEN requirements
- [ ] **Cost Optimization**: Demonstrate savings from two-stage processing
- [ ] **Audit Readiness**: Create examination-ready documentation
- [ ] **Quality Assurance**: Validate AI decisions with human oversight
- [ ] **Scalability**: Design for processing larger datasets
- [ ] **Production Readiness**: Complete testing validates system reliability

## 🎯 Success Criteria

By completion, your integrated system should:
- ✅ Process real financial data with proper validation
- ✅ Execute complete two-stage AI workflow with human gates
- ✅ Generate regulatory-compliant SAR documents
- ✅ Create comprehensive audit trails for all decisions
- ✅ Demonstrate measurable cost optimization benefits
- ✅ Handle errors gracefully and provide clear user feedback
- ✅ Pass all integration and end-to-end tests
- ✅ Meet production-ready quality standards

## 🚀 Next Steps

1. **Complete Implementation**: Fill in all TODO sections with working code
2. **Run Integration Tests**: Validate all components work together properly
3. **Execute End-to-End Tests**: Test complete workflow with automated scenarios
4. **Test Thoroughly**: Run complete workflow with various manual scenarios
5. **Validate Outputs**: Ensure SAR documents meet regulatory requirements
6. **Document Results**: Create final project documentation and metrics
7. **Prepare Presentation**: Demonstrate your system's capabilities and business value

**Congratulations on building a complete AI-powered SAR processing system! 🎉**

## 🧪 Step 7: Workflow Testing and Validation

Before finalizing your implementation, validate your complete system with comprehensive testing.

In [ ]:
# 🧪 Workflow Integration Testing
# Validate your complete system with integration tests

import sys
import os

# Add tests directory to Python path for importing test modules
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)

print(f"📁 Added tests directory to Python path: {tests_path}")

def run_integration_tests():
    """
    Run comprehensive integration tests to validate the complete workflow
    
    Tests include:
    1. Foundation components integration
    2. Agent communication and data flow
    3. End-to-end workflow execution
    4. Error handling and edge cases
    5. Output validation and compliance
    """
    print("🧪 Comprehensive Integration Testing")
    print("📋 TODO: Uncomment and run after implementing complete workflow")
    
    # Uncomment when your complete system is ready:
    try:
        # Import all test modules
        from test_foundation import TestCustomerData, TestAccountData, TestTransactionData, TestCaseData
        from test_risk_analyst import TestRiskAnalystAgent
        from test_compliance_officer import TestComplianceOfficerAgent
        import pytest
        
        print("🔍 Running Foundation Component Tests...")
        foundation_result = pytest.main([
            f"{tests_path}/test_foundation.py", 
            "-v", 
            "--tb=short"
        ])
        
        print("🔍 Running Risk Analyst Agent Tests...")
        risk_result = pytest.main([
            f"{tests_path}/test_risk_analyst.py", 
            "-v", 
            "--tb=short"
        ])
        
        print("📝 Running Compliance Officer Agent Tests...")
        compliance_result = pytest.main([
            f"{tests_path}/test_compliance_officer.py", 
            "-v", 
            "--tb=short"
        ])
        
        # Calculate overall test results
        all_passed = foundation_result == 0 and risk_result == 0 and compliance_result == 0
        
        print("\n" + "="*60)
        print("📊 INTEGRATION TEST RESULTS:")
        print(f"   Foundation Components: {'✅ PASS' if foundation_result == 0 else '❌ FAIL'}")
        print(f"   Risk Analyst Agent: {'✅ PASS' if risk_result == 0 else '❌ FAIL'}")
        print(f"   Compliance Officer Agent: {'✅ PASS' if compliance_result == 0 else '❌ FAIL'}")
        print(f"   Overall Status: {'✅ ALL TESTS PASSED' if all_passed else '❌ SOME TESTS FAILED'}")
        
        if all_passed:
            print("\n🎉 Your system is ready for production workflow testing!")
            print("📝 Proceed to run the complete system demonstration.")
        else:
            print("\n⚠️ Fix failing tests before running the complete workflow.")
            print("📝 Return to previous notebooks to fix component issues.")
        
        return all_passed
            
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure all components are implemented:")
        print("   • foundation_sar.py")
        print("   • risk_analyst_agent.py") 
        print("   • compliance_officer_agent.py")
        return False

def validate_workflow_components():
    """Validate that all required components are available for integration"""
    print("🔍 Validating Workflow Components")
    
    components_status = {
        'foundation_sar': False,
        'risk_analyst_agent': False,
        'compliance_officer_agent': False,
        'test_modules': False
    }
    
    try:
        # Check foundation components
        from foundation_sar import CustomerData, CaseData, ExplainabilityLogger, DataLoader
        components_status['foundation_sar'] = True
        print("✅ Foundation components available")
    except ImportError:
        print("❌ Foundation components not available")
    
    try:
        # Check risk analyst agent
        from risk_analyst_agent import RiskAnalystAgent
        components_status['risk_analyst_agent'] = True
        print("✅ Risk Analyst Agent available")
    except ImportError:
        print("❌ Risk Analyst Agent not available")
    
    try:
        # Check compliance officer agent
        from compliance_officer_agent import ComplianceOfficerAgent
        components_status['compliance_officer_agent'] = True
        print("✅ Compliance Officer Agent available")
    except ImportError:
        print("❌ Compliance Officer Agent not available")
    
    try:
        # Check test modules
        from test_foundation import TestCustomerData
        from test_risk_analyst import TestRiskAnalystAgent  
        from test_compliance_officer import TestComplianceOfficerAgent
        components_status['test_modules'] = True
        print("✅ Test modules available")
    except ImportError:
        print("❌ Test modules not available")
    
    all_ready = all(components_status.values())
    
    print(f"\n📊 Component Status: {'✅ ALL READY' if all_ready else '⚠️ INCOMPLETE'}")
    if not all_ready:
        print("💡 Complete missing components before running integration tests")
    
    return all_ready

# Run component validation
components_ready = validate_workflow_components()

# Run integration tests if components are ready
if components_ready:
    print("\n🚀 All components ready - you can run integration tests!")
    run_integration_tests()
else:
    print("\n📋 Complete component implementation first, then run integration tests")

## Project Walkthrough (Quick)

This run demonstrates the full SAR pipeline:

1. Load CSV data and normalize missing values.
2. Screen for high-risk customers using volume, frequency, and risk rating.
3. Run Risk Analyst (Chain-of-Thought) to classify suspicious activity.
4. Human gate decides whether to proceed with SAR filing.
5. Run Compliance Officer (ReACT) to generate the SAR narrative.
6. Save SAR documents and audit logs.
7. Summarize outcomes and efficiency metrics.

Outputs:
- Audit trail: `../outputs/audit_logs/workflow_integration.jsonl`
- SAR documents: `../outputs/filed_sars/`


In [ ]:
# 🎯 End-to-End Workflow Testing
# Test the complete workflow with known test scenarios

def test_complete_workflow():
    """
    Test the complete SAR processing workflow end-to-end
    
    This test should:
    1. Load test data (can use a subset of actual data)
    2. Run customer screening
    3. Execute two-stage AI analysis
    4. Simulate human decisions (automated for testing)
    5. Generate SAR documents
    6. Validate all outputs
    7. Check audit trail completeness
    """
    print("🎯 End-to-End Workflow Testing")
    print("📋 TODO: Implement after completing all workflow components")
    
    # Example end-to-end test structure (uncomment when ready):
    try:
        print("🚀 Starting end-to-end workflow test...")
        
        # Test data preparation
        print("📊 Loading test data...")
        customers_data, accounts_data, transactions_data = load_and_preprocess_data()
        
        if not customers_data:
            print("⚠️ No test data available - implement data loading first")
            return False
        
        # Test customer screening
        print("🔍 Testing customer screening...")
        selected_customers = screen_high_risk_customers(
            customers_data, accounts_data, transactions_data, top_n=2
        )
        
        if not selected_customers:
            print("⚠️ No customers selected - check screening criteria")
            return False
        
        print(f"✅ Selected {len(selected_customers)} customers for testing")
        
        # Test workflow with automated decisions (for testing)
        print("🤖 Testing automated workflow...")
        test_results = {
            'cases_processed': 0,
            'sars_generated': 0,
            'errors': []
        }
        
        for customer_data in selected_customers:
            try:
                # Create case
                loader = DataLoader(explainability_logger)
                case_data = loader.create_case_from_data(
                    customer_data['customer'],
                    customer_data['accounts'],
                    customer_data['transactions']
                )
                
                # Test Risk Analyst
                risk_analysis = risk_agent.analyze_case(case_data)
                assert hasattr(risk_analysis, 'classification'), "Risk analysis missing classification"
                assert hasattr(risk_analysis, 'confidence_score'), "Risk analysis missing confidence"
                
                # Test Compliance Officer (simulate approval)
                compliance_review = compliance_agent.generate_compliance_narrative(case_data, risk_analysis)
                assert hasattr(compliance_review, 'narrative'), "Compliance review missing narrative"
                
                # Test SAR generation
                sar_document = create_sar_document(case_data, risk_analysis, compliance_review)
                assert sar_document, "SAR document generation failed"
                
                test_results['cases_processed'] += 1
                test_results['sars_generated'] += 1
                
                print(f"✅ Successfully processed: {customer_data['customer']['name']}")
                
            except Exception as e:
                test_results['errors'].append(f"Error processing {customer_data['customer']['name']}: {e}")
                print(f"❌ Error processing: {customer_data['customer']['name']}: {e}")
        
        # Test results summary
        print("\n📊 END-TO-END TEST RESULTS:")
        print(f"   Cases Processed: {test_results['cases_processed']}")
        print(f"   SARs Generated: {test_results['sars_generated']}")
        print(f"   Errors: {len(test_results['errors'])}")
        
        if test_results['errors']:
            print("❌ Test Errors:")
            for error in test_results['errors']:
                print(f"     • {error}")
        
        success = len(test_results['errors']) == 0 and test_results['cases_processed'] > 0
        
        if success:
            print("\n🎉 END-TO-END TEST PASSED!")
            print("✅ Your complete workflow is ready for production use!")
        else:
            print("\n⚠️ END-TO-END TEST HAD ISSUES")
            print("📝 Fix the errors above before deploying to production")
        
        return success
        
    except Exception as e:
        print(f"❌ End-to-end test failed: {e}")
        print("💡 Ensure all components are properly implemented")
        return False

# Run end-to-end test
print("📋 TODO: Run end-to-end test after implementing complete workflow")
test_success = test_complete_workflow()